In [ ]:
import os, glob, re
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SERIES_COLS = [
    "J_n_sim_cumulative",
    "J_real_cumulative",
    "abs_gap_cumulative",
    "proxy_bound_cumulative",
]

def parse_array(s) -> np.ndarray:
    """Parse strings like '[0.1 0.2 0.3]' or '[0.1, 0.2, 0.3]' into np.array."""
    s = str(s).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]
    s = re.sub(r"[,\n\r\t]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if not s:
        return np.array([], dtype=float)
    return np.fromstring(s, sep=" ", dtype=float)

def load_one_csv(csv_path: str) -> Dict[str, np.ndarray]:
    df = pd.read_csv(csv_path)
    if df.empty:
        raise ValueError(f"Empty CSV: {csv_path}")

    if "split" in df.columns:
        test_rows = df[df["split"].astype(str).str.lower() == "test"]
        row = test_rows.iloc[0] if len(test_rows) else df.iloc[0]
    else:
        row = df.iloc[0]

    out = {}
    for c in SERIES_COLS:
        if c not in df.columns:
            raise KeyError(f"Missing column {c} in {csv_path}")
        out[c] = parse_array(row[c])

    out["gap_check"] = np.abs(out["J_n_sim_cumulative"] - out["J_real_cumulative"])
    return out

def stack_trim(arrs: List[np.ndarray]) -> np.ndarray:
    """Stack list of 1D arrays to (R,T) by trimming all to min length."""
    lens = [len(a) for a in arrs if a is not None and len(a) > 0]
    if not lens:
        raise ValueError("No non-empty arrays to stack.")
    T = min(lens)
    X = np.vstack([a[:T] for a in arrs])
    return X

def pick_k_folders(root: str, k: int, random_seed: int = 42) -> List[str]:
    run_dirs = [d for d in glob.glob(os.path.join(root, "*")) if os.path.isdir(d)]
    run_dirs.sort()
    rng = np.random.default_rng(random_seed)
    k = min(k, len(run_dirs))
    return rng.choice(run_dirs, size=k, replace=False).tolist()

# mean + 95% CI
def plot_scr_ccm_paper_figure(
    J_sim_runs: np.ndarray, 
    J_real_runs: np.ndarray, 
    gap_runs: np.ndarray, 
    bound_runs: np.ndarray,  
    out_prefix: str = "fig_scr_ccm",
    start_day_zoom: int = 1,
    eps: float = 1e-12,
):
    assert J_sim_runs.shape == J_real_runs.shape == gap_runs.shape == bound_runs.shape
    R, T = J_sim_runs.shape

    start_idx = max(1, int(start_day_zoom)) - 1
    sl = slice(start_idx, None)
    t = np.arange(1, T + 1)[sl]

    def mean_ci95(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        X = X[:, sl]
        mu = X.mean(axis=0)
        sd = X.std(axis=0, ddof=1)
        se = sd / np.sqrt(R)
        ci = 1.96 * se
        return mu, ci

    mu_sim,  ci_sim  = mean_ci95(J_sim_runs)
    mu_real, ci_real = mean_ci95(J_real_runs)

    ratio_runs = (gap_runs[:, sl]) / (bound_runs[:, sl] + eps)
    mu_ratio = ratio_runs.mean(axis=0)
    sd_ratio = ratio_runs.std(axis=0, ddof=1)
    ci_ratio = 1.96 * (sd_ratio / np.sqrt(R))

    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 11,
    })

    fig, ax1 = plt.subplots(1, 1, figsize=(6.5, 3), sharex=True)

    # Cumulative returns
    ax1.plot(t, mu_sim, lw=2, label=r"$J_{scen}$ (mean)")
    c1 = ax1.lines[-1].get_color()
    ax1.fill_between(t, mu_sim - ci_sim, mu_sim + ci_sim,
                     color=c1, alpha=0.35, linewidth=0, zorder=1)

    ax1.plot(t, mu_real, lw=2, label=r"$J_{real}$ (mean)")
    c2 = ax1.lines[-1].get_color()
    ax1.fill_between(t, mu_real - ci_real, mu_real + ci_real,
                     color=c2, alpha=0.35, linewidth=0, zorder=1)

    ax1.set_ylabel("Cumulative return")
    ax1.set_title(f"Expected vs. Realized (Cumulative): mean ± 95% CI")
    ax1.legend(frameon=False, loc="best")
    ax1.grid(True, linestyle="--", alpha=0.4)

    low  = min((mu_sim - ci_sim).min(),  (mu_real - ci_real).min())
    high = max((mu_sim + ci_sim).max(),  (mu_real + ci_real).max())
    pad = 0.08 * (high - low + 1e-12)
    ax1.set_ylim(low - pad, high + pad)
    ax1.set_xlabel("Trading days (out-of-sample)")
    plt.tight_layout()

    pdf_path = f"{out_prefix}.pdf"
    png_path = f"{out_prefix}.png"
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.close()


# Load 10 runs from folders and plot
def load_runs_from_folders(
    ROOT: str,
    run_names: Optional[List[str]] = None,
    k: int = 10,
    random_seed: int = 42,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if run_names is None:
        chosen_dirs = pick_k_folders(ROOT, k=k, random_seed=random_seed)
        run_names = [os.path.basename(d) for d in chosen_dirs]

    loaded = []
    for name in run_names:
        csv_path = os.path.join(ROOT, name, "logs", "scr_ppo_full_sim_pac_test.csv")
        if not os.path.isfile(csv_path):
            print("[WARN] missing:", csv_path)
            continue
        try:
            d = load_one_csv(csv_path)
            d["run"] = name
            loaded.append(d)
        except Exception as e:
            print("[WARN] failed:", csv_path, "err:", e)

    if len(loaded) < 2:
        raise RuntimeError(f"Need >=2 runs loaded, got {len(loaded)}")


    J_sim  = stack_trim([d["J_n_sim_cumulative"]     for d in loaded])
    J_real = stack_trim([d["J_real_cumulative"]      for d in loaded])
    gap    = stack_trim([d["gap_check"]              for d in loaded])
    bound  = stack_trim([d["proxy_bound_cumulative"] for d in loaded])

    T = min(J_sim.shape[1], J_real.shape[1], gap.shape[1], bound.shape[1])
    return J_sim[:, :T], J_real[:, :T], gap[:, :T], bound[:, :T]

if __name__ == "__main__":
    ROOT = ""
    run_names = [
        "rl_runs_scr_ccm_seed_0",
        "rl_runs_scr_ccm_seed_1",
        "rl_runs_scr_ccm_seed_2",
        "rl_runs_scr_ccm_seed_3",
        "rl_runs_scr_ccm_seed_4474",
        "rl_runs_scr_ccm_seed_9944",
        "rl_runs_scr_ccm_seed_9925",
        "rl_runs_scr_ccm_seed_3820",
        "rl_runs_scr_ccm_seed_9535",
        "rl_runs_scr_ccm_seed_8271",

    ]
    J_sim, J_real, gap, bound = load_runs_from_folders(ROOT, run_names=run_names, k=10, random_seed=42)
    plot_scr_ccm_paper_figure(
        J_sim_runs=J_sim,
        J_real_runs=J_real,
        gap_runs=gap,
        bound_runs=bound,
        out_prefix="fig_scr_ccm_plot",
        start_day_zoom=200, 
    )

In [ ]:
import os
import numpy as np
import pandas as pd

def load_contraction_csv(csv_path: str, y_col: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if "epoch" not in df.columns:
        raise KeyError(f"'epoch' missing in {csv_path}")
    if y_col not in df.columns:
        raise KeyError(f"'{y_col}' missing in {csv_path}. Have {df.columns.tolist()}")

    df = df[["epoch", y_col]].dropna().copy()
    df["epoch"] = df["epoch"].astype(int)
    df = df.groupby("epoch", as_index=False)[y_col].mean()
    df = df.sort_values("epoch").reset_index(drop=True)
    return df.rename(columns={y_col: "y"})

def load_run_dfs(ROOT: str, run_names: list, y_col: str, filename="scr_ppo_full_contraction_diag.csv"):
    run_dfs = {}
    missing = []
    failed = []

    for rn in run_names:
        p = os.path.join(ROOT, rn, "logs", filename)
        if not os.path.isfile(p):
            missing.append(p)
            continue
        try:
            run_dfs[rn] = load_contraction_csv(p, y_col=y_col)
        except Exception as e:
            failed.append((p, str(e)))

    print(f"Loaded {len(run_dfs)} / {len(run_names)} runs")
    if missing:
        print("\nMissing files:")
        for m in missing: print(m)
    if failed:
        print("\nFailed loads:")
        for p, err in failed:
            print(p)
            print("  err:", err)

    if len(run_dfs) == 0:
        raise RuntimeError("No runs loaded. Check ROOT / run_names / file paths.")

    return run_dfs

# AGGREGATE (log-space CI)
def aggregate_by_epoch_union_logci(run_dfs, min_count=8, eps=1e-12):
    merged = None
    for rn, df in run_dfs.items():
        s = df.set_index("epoch")["y"].rename(rn)
        merged = s.to_frame() if merged is None else merged.join(s, how="outer")
    merged = merged.sort_index()
    count = merged.notna().sum(axis=1)
    merged = merged[count >= min_count]
    count = count[count >= min_count]
    if merged.empty:
        raise RuntimeError(f"No epochs with >= {min_count} runs. Lower min_count or check data.")

    X = merged.to_numpy(float)
    X = np.clip(X, eps, None)
    Z = np.log(X)

    mu_z = np.nanmean(Z, axis=1)
    sd_z = np.nanstd(Z, axis=1, ddof=1)
    se_z = sd_z / np.sqrt(count.to_numpy(float))
    ci_z = 1.96 * se_z

    out = pd.DataFrame({
        "epoch": merged.index.to_numpy(),
        "mean": np.exp(mu_z),
        "lo":   np.exp(mu_z - ci_z),
        "hi":   np.exp(mu_z + ci_z),
        "count": count.to_numpy(),
    })
    return out

# Plot
import matplotlib.pyplot as plt


def plot_contraction_logband(agg, title, y_label, out_prefix, x_mode="progress"):
    e = agg["epoch"].to_numpy()
    mu = agg["mean"].to_numpy()
    lo = agg["lo"].to_numpy()
    hi = agg["hi"].to_numpy()
    if x_mode == "progress":
        denom = e.max() - e.min()
        x = (e - e.min()) / denom if denom > 0 else np.zeros_like(e)
        x_label = "Training progress"
    else:
        x = e
        x_label = "epoch"

    fig, ax = plt.subplots(figsize=(6.5, 3.7))
    ax.plot(x, mu, lw=2, label="mean")
    c = ax.lines[-1].get_color()
    ax.fill_between(x, lo, hi, color=c, alpha=0.30, linewidth=0, label="95% CI (log-space)")

    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_yscale("log")

    if x_mode == "progress":
        ax.set_xlim(0.0, 1.0)
        ax.set_xticks(np.linspace(0, 1, 6))

    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(frameon=False, loc="best")

    plt.tight_layout()
    plt.savefig(out_prefix + ".pdf", bbox_inches="tight")
    plt.savefig(out_prefix + ".png", dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved:", out_prefix + ".pdf")
    print("Saved:", out_prefix + ".png")



ROOT = ""
run_names = [
    "rl_runs_scr_ccm_seed_0",
    "rl_runs_scr_ccm_seed_1",
    "rl_runs_scr_ccm_seed_2",
    "rl_runs_scr_ccm_seed_3",
    "rl_runs_scr_ccm_seed_4474",
    "rl_runs_scr_ccm_seed_9944",
    "rl_runs_scr_ccm_seed_9925",
    "rl_runs_scr_ccm_seed_3820",
    "rl_runs_scr_ccm_seed_9535",
    "rl_runs_scr_ccm_seed_8271",

]


y_col =  "resid_l2"

run_dfs = load_run_dfs(ROOT, run_names, y_col=y_col)
agg = aggregate_by_epoch_union_logci(run_dfs, min_count=8)

plot_contraction_logband(
    agg,
    title=f"Learning Stability (Contraction): {y_col}",
    y_label=f"Bellman residual ({y_col})",
    out_prefix=f"fig2_contraction_{y_col}_logCI"
)